In [ ]:
!nvidia-smi -L
!apt-get update -qq && apt-get install -y -qq libegl1 libgl1 libxkbcommon0 libsm6 libxi6 libxxf86vm1 libxfixes3 libxrender1 > /dev/null
!pip install -q bpy
import bpy; print('Blender', bpy.app.version_string)

In [ ]:
import glob, os, zipfile, subprocess, time, re, shutil
PART_LO, PART_HI = 76, 150   # which frames this notebook renders
K = '/kaggle/temp/kit'; os.makedirs(K, exist_ok=True)
z = glob.glob('/kaggle/input/**/velo_kit.zip', recursive=True)
if z: zipfile.ZipFile(z[0]).extractall(K)
else:
    b = glob.glob('/kaggle/input/**/base_rig.blend', recursive=True)[0]; shutil.copytree(os.path.dirname(b), K, dirs_exist_ok=True)
print(os.listdir(K))
subprocess.run(['python3', f'{K}/tools/assemble.py', 'clip', f'{K}/examples/hero_test_section.py', '--dur', '5', '--out', f'{K}/clip_v1.py'], check=True)
s = open(f'{K}/clip_v1.py').read()
# Cycles on the GPU (EEVEE can't use these GPUs), 720x1280, 32 samples + denoise; render ALL frames of the 5 s clip
s = s.replace("    try: sc.eevee.taa_render_samples = 24\n    except Exception: pass\n", '''    sc.render.engine = 'CYCLES'; sc.cycles.samples = 16; sc.cycles.use_denoising = True; sc.cycles.use_adaptive_sampling = True
    _cp = bpy.context.preferences.addons['cycles'].preferences; _dev = 'CPU'
    for _t in ('OPTIX', 'CUDA'):
        try:
            _cp.compute_device_type = _t; _cp.get_devices()
            _g = [d for d in _cp.devices if d.type != 'CPU']
            if _g:
                for d in _cp.devices: d.use = d.type != 'CPU'
                sc.cycles.device = 'GPU'; _dev = _t + ' ' + ', '.join(d.name for d in _g); break
        except Exception as _e: log('device try failed', _t, _e)
    log('CYCLES DEVICE', _dev)
    r.resolution_x, r.resolution_y = 720, 1280; r.resolution_percentage = 100; r.use_persistent_data = True
''', 1)
s = s.replace("    if RUNMODE == 'stills':", '''    if RUNMODE == 'video':
        r.image_settings.file_format = 'PNG'; FD = os.path.join(HERE, 'clipframes'); os.makedirs(FD, exist_ok=True)
        T_ = time.time()
        for f_ in [x for x in FRAMES if PART_LO <= x <= PART_HI]:
            r.filepath = os.path.join(FD, '%04d.png' % f_); sc.frame_set(f_); bpy.ops.render.render(write_still=True)
            if f_ in (1, 2, 5, 10) or f_ % 25 == 0: log('FRAME', f_, 'of', len(FRAMES), 'elapsed', round(time.time() - T_, 1))
    if RUNMODE == 'stills':''', 1)
s = 'PART_LO, PART_HI = %d, %d\n' % (PART_LO, PART_HI) + s
open(f'{K}/clip_v1.py', 'w').write(s)
open(f'{K}/clip_mode.txt', 'w').write('video')

In [ ]:
t = time.time()
r = subprocess.run(f'cd {K} && python3 tools/run_local.py {K} clip', shell=True, capture_output=True, text=True)
print('SECONDS', round(time.time() - t))
print(open(f'{K}/log_clip.txt').read()[-2500:])
print(r.stderr[-600:])

In [ ]:
os.makedirs('/kaggle/working', exist_ok=True)
fr = sorted(glob.glob(f'{K}/clipframes/*.png')); print('frames', len(fr))
shutil.copy(f'{K}/log_clip.txt', '/kaggle/working/log_clip.txt')
shutil.make_archive('/kaggle/working/frames', 'zip', f'{K}/clipframes')
print(os.listdir('/kaggle/working'))